# Vacation Recommendations

> **File:** `vacations.ipynb`  
> **Data:** `resources/cities_weather.csv`  
> **Purpose:** The application of a set of specified weather conditions to filter the global dataset down to a shortlist of candidate vacation destinations

---

## Executive Summary

This notebook is a companion to `weather.ipynb`. It takes the city weather dataset produced by that notebook — **714 cities** sampled globally via `citipy` and the OpenWeatherMap API — and narrows them down to locations that meet a specific set of ideal holiday conditions. For each qualifying city it then queries the Geoapify Places API to find a nearby hotel, restaurant, and tourist attraction, and plots everything on an interactive map.

### Notebook Structure

| Section | Content |
|---------|---------|
| 1 | Import city weather CSV; display all cities on a map |
| 2 | Apply weather filters; display qualifying cities |
| 3 | Query Geoapify for nearest hotel per city; map results |
| 4 | Query Geoapify for nearest restaurant per city; map results |
| 5 | Query Geoapify for nearest tourist attraction per city; final map |

### Ideal Weather Criteria

The filter applied in Section 2 is deliberately narrow:

| Condition | Range |
|-----------|-------|
| Temperature | 70–95 °F |
| Humidity | 35–65% |
| Cloudiness | 0–10% |
| Wind speed | 0–10 m/s |

These four filters applied in sequence reduce the 714-city dataset significantly — temperature alone cuts it to 286 cities, humidity to 55, and the near-clear-sky cloudiness requirement to 9. The wind speed filter leaves **3 qualifying cities**.

### Qualifying Cities

Based on the `cities_weather.csv` data collected on 18 March 2026, exactly three cities meet all conditions:

| City | Country | Temp (°F) | Humidity (%) | Cloudiness (%) | Wind (m/s) |
|------|---------|-----------|--------------|----------------|------------|
| Changem | India | 87.8 | 45 | 5 | 4.6 |
| Avenal | United States | 72.1 | 57 | 1 | 4.5 |
| Myeik | Myanmar | 86.9 | 60 | 5 | 8.4 |

The cloudiness threshold (≤ 10%) is the most restrictive filter by far: only 9 of 714 cities — just 1.3% of the sample — have near-clear skies at the time of data collection. This reflects the snapshot nature of the dataset; results will differ on every run.

### What the Notebook Produces

For each qualifying city the notebook calls the Geoapify Places API to locate the nearest hotel, restaurant, and tourist attraction, appending each to the city record. The final output is a series of interactive Folium maps — one per enrichment stage — with tooltips showing the venue name, city, and country. The final map (Figure 5.4) shows all three layers together.

---


In [1]:
 #*******************************************************************************************
 #
 #  File Name:  vacations.ipynb 
 #
 #  File Description:
 #      This Jupyter Notebook, vacations.ipynb, is a Python script to determine the 
 #      ideal locations (city and hotel) for a vacation and displays information on 
 #      a map.
 #      
 #
 #  Date                Description                                 Programmer
 #  ---------------     ------------------------------------        ------------------
 #  08/12/2023          Initial Development                         Nicholas J. George
 #  03/04/2026          Upgraded Module                             Nicholas J. George
 #
 #******************************************************************************************/

import citipyx
import mapx

import logx
import pandasx

import warnings

import pandas as pd

from bokeh.util.warnings import BokehUserWarning


warnings.filterwarnings('ignore')

warnings.simplefilter(action = 'ignore', category = BokehUserWarning)


pd.options.mode.chained_assignment = None

In [2]:
CONSTANT_LOCAL_FILE_NAME = 'vacations.ipynb'

In [3]:
logx.set_log_mode(False)

logx.set_image_mode(False)


logx.begin_program('vacations')

Program execution begins...



In [4]:
citipyx.set_vac_temp_rng(70, 95)

citipyx.set_vac_humid_rng(35, 65)

citipyx.set_vac_cloud_rng(0, 10)

citipyx.set_vac_wind_speed_rng(0, 10)

# <br> **Section 1: Vacation Data Acquisition**

## **1.1: Data Import from CSV File**

In [5]:
city_weather_df \
    = pd.read_csv \
        (citipyx.config_dict['data']['datafile'],
         index_col = citipyx.config_dict['params']['index'])

logx.log_write_obj(city_weather_df)

## **1.2: Display City Weather Data Set**

In [6]:
pandasx.rtn_fmt_tbl(city_weather_df, 'Table: 1.2: City Weather Information')

city,latitude,longitude,temperature,humidity,cloudiness,wind_speed,country,date_time
college,64.86,-147.80,-2.43,52,100,3.44,US,2026-03-19 14:21:41
bredasdorp,-34.53,20.04,68.11,77,39,5.26,ZA,2026-03-19 14:21:41
natal port,-51.72,-72.49,54.05,54,20,6.91,CL,2026-03-19 14:21:42
dudinka,69.41,86.18,-33.05,100,85,3.06,RU,2026-03-19 14:21:43
beautiful,69.22,-51.10,-0.38,78,100,13.80,GL,2026-03-19 14:21:44
puerto madero,14.72,-92.42,89.80,58,40,11.50,MX,2026-03-19 14:21:44
nuuk,64.18,-51.72,5.61,72,75,16.11,GL,2026-03-19 14:21:45
talagutong,6.26,125.67,78.31,83,54,9.60,PH,2026-03-19 14:21:46
waitangi,-43.95,-176.56,58.82,94,100,5.01,NZ,2026-03-19 14:21:46
talnakh,69.49,88.40,-12.37,99,100,1.07,RU,2026-03-19 14:21:47


## **1.3: Display City Weather Locations**

In [7]:
mapx.set_tooltip_display(False)

mapx.disp_folium_circles_df(city_weather_df, 'Figure 1.3: City Weather Locations')

# <br> **Section 2: Desired Weather Locations**

## **2.1: Establish Desired Weather Conditions for Vacation Locations**

In [8]:
weather_dict = citipyx.get_weather_dict()

vacations_df \
    = city_weather_df \
        .loc[(city_weather_df['temperature'] \
                >= weather_dict['min_temp']) \
             & (city_weather_df['temperature'] \
                <= weather_dict['max_temp']), :]

vacations_df \
    = vacations_df \
        .loc[(vacations_df['humidity'] \
                >= weather_dict['min_humid']) \
             & (vacations_df['humidity'] \
                <= weather_dict['max_humid']), :]

vacations_df \
    = vacations_df \
        .loc[(vacations_df['cloudiness'] \
                >= weather_dict['min_cloud']) \
             & (vacations_df['cloudiness'] \
                <= weather_dict['max_cloud']), :]

vacations_df \
    = vacations_df \
        .loc[(vacations_df['wind_speed'] \
                >= weather_dict['min_wind_speed']) 
             & (vacations_df['wind_speed'] \
                <= weather_dict['max_wind_speed']), :]

vacations_df.dropna(inplace = True)

vacations_df.reset_index(drop = True, inplace = True)


logx.log_write_obj(vacations_df)

## **2.2: Display Vacation Data Set**

In [9]:
pandasx.rtn_fmt_tbl(vacations_df, 'Table: 2.3: Vacation Locations')

city,latitude,longitude,temperature,humidity,cloudiness,wind_speed,country,date_time
karratha,-20.74,116.85,80.49,63,3,7.43,AU,2026-03-19 14:22:06
sahar,24.36,56.75,80.62,50,0,0.00,OM,2026-03-19 14:22:30
san luis de la loma,17.27,-100.89,84.69,57,0,8.95,MX,2026-03-19 14:23:51
shenandoah,30.40,-91.00,76.26,44,3,5.01,US,2026-03-19 14:24:01
coahuayana de hidalgo,18.70,-103.66,87.26,45,0,7.67,MX,2026-03-19 14:25:26
fortuna,40.60,-124.16,73.92,58,5,3.00,US,2026-03-19 14:25:58
pacific grove,36.62,-121.92,84.27,46,1,7.92,US,2026-03-19 14:27:42
guanica,17.97,-66.91,82.76,65,2,7.00,PR,2026-03-19 14:27:56
on,22.57,59.53,79.92,49,7,9.01,OM,2026-03-19 14:28:29
demopolis,32.52,-87.84,73.33,38,9,4.16,US,2026-03-19 14:29:09


## **2.3: Display Vacation Locations**

In [10]:
mapx.set_tooltip_display(True)

mapx.disp_folium_circles_df(vacations_df, 'Figure 2.4: Vacation Locations')

# <br> **Section 3: Hotel Locations**

## **3.1: Add Hotel Column to DataFrame**

In [11]:
hotels_df = vacations_df.copy()

hotels_df['hotel_name'] = pd.Series(dtype = 'str')

hotels_df.reset_index(drop = True, inplace = True)

logx.log_write_obj(hotels_df)

## **3.2: Find Hotel Locations**

In [12]:
updated_hotels_df \
    = citipyx.upd_loc_vac_df \
        (hotels_df, 
         'hotel_name', 
         'accommodation.hotel')

logx.log_write_obj(updated_hotels_df)

STARTING HOTEL SEARCH...


Located the following hotel...Karratha International Hotel in karratha, AU


Located the following hotel...Marina Sohar in karratha, AU


Located the following hotel...La Quinta Inn & Suites Baton Rouge Denham Springs in karratha, AU


Located the following hotel...Comfort Inn & Suites Redwood Country in karratha, AU


Located the following hotel...Pacific Grove Inn in karratha, AU


Located the following hotel...Copamarina Beach Resort in karratha, AU


Located the following hotel...Sur Hotel in karratha, AU


Located the following hotel...Hermosina Hotel in karratha, AU


Located the following hotel...Andalusia Hotel in karratha, AU


Located the following hotel...646 Hotel Balcarce in karratha, AU


HOTEL SEARCH COMPLETE...




## **3.3: Display Hotel Data Set**

In [13]:
pandasx.rtn_fmt_tbl(updated_hotels_df, 'Table: 3.3: Hotel Locations')

city,latitude,longitude,temperature,humidity,cloudiness,wind_speed,country,date_time,hotel_name
karratha,-20.74,116.85,80.49,63,3,7.43,AU,2026-03-19 14:22:06,karratha international hotel
sahar,24.36,56.75,80.62,50,0,0.00,OM,2026-03-19 14:22:30,marina sohar
shenandoah,30.40,-91.00,76.26,44,3,5.01,US,2026-03-19 14:24:01,la quinta inn & suites baton rouge denham springs
fortuna,40.60,-124.16,73.92,58,5,3.00,US,2026-03-19 14:25:58,comfort inn & suites redwood country
pacific grove,36.62,-121.92,84.27,46,1,7.92,US,2026-03-19 14:27:42,pacific grove inn
guanica,17.97,-66.91,82.76,65,2,7.00,PR,2026-03-19 14:27:56,copamarina beach resort
on,22.57,59.53,79.92,49,7,9.01,OM,2026-03-19 14:28:29,sur hotel
chilecito,-29.16,-67.50,81.52,44,1,8.86,AR,2026-03-19 14:29:22,hermosina hotel
arica,-18.48,-70.30,75.43,60,0,8.05,CL,2026-03-19 14:29:43,andalusia hotel
balcarce,-37.85,-58.26,77.83,44,0,5.01,AR,2026-03-19 14:30:33,646 hotel balcarce


## **3.4: Display Hotel Locations**

In [14]:
mapx.set_tooltip_cols(['hotel_name', 'city', 'country'])

mapx.disp_folium_circles_df(updated_hotels_df, 'Figure 3.4: Hotel Locations')

# <br> **Section 4: Restaurant Locations**

## **4.1: Add Restaurant Column to DataFrame**

In [15]:
restaurant_df = updated_hotels_df.copy()

restaurant_df['restaurant_name'] = pd.Series(dtype = 'str')

restaurant_df.reset_index(drop = True, inplace = True)

logx.log_write_obj(restaurant_df)

## **4.2: Find Restaurant Locations**

In [16]:
upd_restaurant_df \
    = citipyx.upd_loc_vac_df \
        (restaurant_df, 
         'restaurant_name', 
         'catering.restaurant')

logx.log_write_obj(upd_restaurant_df)

STARTING RESTAURANT SEARCH...


Located the following restaurant...Light Bar and Food in karratha, AU


Located the following restaurant...Mashaer7 in karratha, AU


Located the following restaurant...Kabob's Lebanese & Greek in karratha, AU


Located the following restaurant...Hunan Village in sahar, OM


Located the following restaurant...Petra Restaurant in karratha, AU


Located the following restaurant...Tomatoes in karratha, AU


Located the following restaurant...Kunnath Restaurant in karratha, AU


Located the following restaurant...El Chacho in sahar, OM


Located the following restaurant...The Ganguita in karratha, AU


Located the following restaurant...Cafe Fangio in karratha, AU


RESTAURANT SEARCH COMPLETE...




## **4.3: Display Restaurant Data Set**

In [17]:
pandasx.rtn_fmt_tbl(upd_restaurant_df, 'Table: 4.3: Restaurant Locations')

city,latitude,longitude,temperature,humidity,cloudiness,wind_speed,country,date_time,hotel_name,restaurant_name
karratha,-20.74,116.85,80.49,63,3,7.43,AU,2026-03-19 14:22:06,karratha international hotel,light bar and food
sahar,24.36,56.75,80.62,50,0,0.00,OM,2026-03-19 14:22:30,marina sohar,mashaer7
shenandoah,30.40,-91.00,76.26,44,3,5.01,US,2026-03-19 14:24:01,la quinta inn & suites baton rouge denham springs,kabob's lebanese & greek
fortuna,40.60,-124.16,73.92,58,5,3.00,US,2026-03-19 14:25:58,comfort inn & suites redwood country,hunan village
pacific grove,36.62,-121.92,84.27,46,1,7.92,US,2026-03-19 14:27:42,pacific grove inn,petra restaurant
guanica,17.97,-66.91,82.76,65,2,7.00,PR,2026-03-19 14:27:56,copamarina beach resort,tomatoes
on,22.57,59.53,79.92,49,7,9.01,OM,2026-03-19 14:28:29,sur hotel,kunnath restaurant
chilecito,-29.16,-67.50,81.52,44,1,8.86,AR,2026-03-19 14:29:22,hermosina hotel,el chacho
arica,-18.48,-70.30,75.43,60,0,8.05,CL,2026-03-19 14:29:43,andalusia hotel,the ganguita
balcarce,-37.85,-58.26,77.83,44,0,5.01,AR,2026-03-19 14:30:33,646 hotel balcarce,cafe fangio


## **4.4: Display Restaurant Locations**

In [18]:
mapx.set_tooltip_cols(['hotel_name', 'restaurant_name', 'city', 'country'])

mapx.disp_folium_circles_df(upd_restaurant_df, 'Figure 4.4: Restaurant Locations')

# <br> **Section 5: Tourism Attraction Locations**

## **5.1: Add Tourism Attraction Column to DataFrame**

In [19]:
tourist_attraction_df = upd_restaurant_df.copy()

tourist_attraction_df['tourist_attraction'] = pd.Series(dtype = 'str')

tourist_attraction_df.reset_index(drop = True, inplace = True)


logx.log_write_obj(tourist_attraction_df)

## **5.2: Find Tourism Attraction Locations**

In [20]:
upd_tourist_attract_df \
    = citipyx.upd_loc_vac_df \
        (tourist_attraction_df, 
         'tourist_attraction', 
         'tourism.attraction')

logx.log_write_obj(upd_tourist_attract_df)

STARTING TOURISM ATTRACTION SEARCH...


Located the following tourism attraction...Karratha Water Tanks Artwork in karratha, AU


Located the following tourism attraction...Trust Restaurant in karratha, AU


Located the following tourism attraction...Louisiana Mud Painting Gallery in sahar, OM


Located the following tourism attraction...Giant Wooden Cross in karratha, AU


Located the following tourism attraction...Lover's Point Mural in karratha, AU


Located the following tourism attraction...Fort Capron in karratha, AU


Located the following tourism attraction...Naama Coast in karratha, AU


Located the following tourism attraction...Christ of Portezuelo in sahar, OM


Located the following tourism attraction...HipHop in karratha, AU


Located the following tourism attraction...Deer in karratha, AU


TOURISM ATTRACTION SEARCH COMPLETE...




## **5.3: Display Tourism Attraction Data Set**

In [21]:
pandasx.rtn_fmt_tbl(upd_tourist_attract_df, 'Table: 5.3: Tourist Attraction Locations')

city,latitude,longitude,temperature,humidity,cloudiness,wind_speed,country,date_time,hotel_name,restaurant_name,tourist_attraction
karratha,-20.74,116.85,80.49,63,3,7.43,AU,2026-03-19 14:22:06,karratha international hotel,light bar and food,karratha water tanks artwork
sahar,24.36,56.75,80.62,50,0,0.00,OM,2026-03-19 14:22:30,marina sohar,mashaer7,trust restaurant
shenandoah,30.40,-91.00,76.26,44,3,5.01,US,2026-03-19 14:24:01,la quinta inn & suites baton rouge denham springs,kabob's lebanese & greek,louisiana mud painting gallery
fortuna,40.60,-124.16,73.92,58,5,3.00,US,2026-03-19 14:25:58,comfort inn & suites redwood country,hunan village,giant wooden cross
pacific grove,36.62,-121.92,84.27,46,1,7.92,US,2026-03-19 14:27:42,pacific grove inn,petra restaurant,lover's point mural
guanica,17.97,-66.91,82.76,65,2,7.00,PR,2026-03-19 14:27:56,copamarina beach resort,tomatoes,fort capron
on,22.57,59.53,79.92,49,7,9.01,OM,2026-03-19 14:28:29,sur hotel,kunnath restaurant,naama coast
chilecito,-29.16,-67.50,81.52,44,1,8.86,AR,2026-03-19 14:29:22,hermosina hotel,el chacho,christ of portezuelo
arica,-18.48,-70.30,75.43,60,0,8.05,CL,2026-03-19 14:29:43,andalusia hotel,the ganguita,hiphop
balcarce,-37.85,-58.26,77.83,44,0,5.01,AR,2026-03-19 14:30:33,646 hotel balcarce,cafe fangio,deer


## **5.4: Display Tourism Attraction Locations**

In [22]:
mapx.set_tooltip_cols(['hotel_name', 'restaurant_name', 'tourist_attraction', 'city', 'country'])

mapx.disp_folium_circles_df(upd_tourist_attract_df, 'Figure 5.4: Tourist Attraction Locations')

In [23]:
# logx.end_program()